### Bloco 0: Montar o Google Drive
Célula necessária para permitir que o Colab acesse e grave arquivos diretamente na raiz do seu Google Drive.

In [ ]:
# O pacote 'google-colab' já vem pré-instalado por padrão no ambiente do Google Colab.
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive montado com sucesso!")
except Exception as e:
    print(f"⚠️ Nota: Se você estiver rodando localmente (fora do Colab), a montagem do Drive pode ser desconsiderada. Erro: {e}")

### Bloco 1: Instalação das Dependências

In [ ]:
!pip install -q google-genai pytelegrambotapi pandas openpyxl soundfile

### Bloco 2: Configurações Iniciais

In [ ]:
import os
import telebot
import pandas as pd
import soundfile as sf
from google import genai
from google.genai import types
from decimal import Decimal, InvalidOperation
import json
import re
from pathlib import Path

# Configurações de API
TELEGRAM_TOKEN = '-----'
GEMINI_API_KEY = '-----'

#PLANILHA_NOME = "/content/drive/-----"
PLANILHA_NOME = Path(r"C:\Users\-----")

# Inicialização das APIs
bot = telebot.TeleBot(TELEGRAM_TOKEN)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Clientes inicializados e apontando para a planilha no caminho correto do Drive!")

### Bloco 3: Prompt do Contrato de Dados (IA)
Definimos o prompt rígido que instrui o `gemini-3.6-flash` a categorizar se a mensagem é para `inserir` ou `consultar` e a estruturar a saída estritamente em formato JSON.

In [ ]:
PROMPT_CONTRATO_DADOS = """
Você é um assistente de vendas altamente preciso. Sua tarefa é analisar o texto enviado pelo usuário e classificar a intenção entre "inserir" ou "consultar".

- Se a intenção for "consultar":
Retorne um JSON detalhado extraindo os parâmetros de busca específicos do texto do usuário. Defina como null as chaves que não forem mencionadas:
{
  "intencao": "consultar",
  "filtros": {
    "cliente": "<nome do cliente para filtrar ou null>",
    "produto": "<descrição do produto vendido para filtrar ou null>",
    "data": "<data específica no formato DD/MM/AAAA ou null>",
    "como_pagou": "<valores possíveis: pix, dinheiro, debito, credito, fiado ou null>",
    "busca_geral": "<qualquer termo de pesquisa genérico que não se encaixe nos outros campos ou null>"
  }
}

- Se a intenção for "inserir":
Retorne APENAS o seguinte JSON, extraindo as informações do texto. Caso não encontre a informação para um campo opcional, defina-o como null (ou de acordo com as regras abaixo):
{
  "intencao": "inserir",
  "dados": {
    "data": "<data no formato DD/MM/AAAA de acordo com a data atual (datetime.now().strftime("%d/%m/%Y")), caso fale ontem e/ou amanhã usar como referencia a data atual e aplica a logica, ou null se não especificado nenhum desse termos>",
    "oq_vendeu": "<descrição do produto vendido ou null>",
    "pra_quem": "<nome do cliente ou null>",
    "valor": <número decimal ou null>,
    "valor_restante": <número decimal ou null para fiado, ou zero/null se pago integralmente>,
    "como_pagou": "<valores possíveis: pix, dinheiro, debito, credito, fiado ou null>",
    "cartao": "<bandeira ou tipo do cartão se aplicável, ou null>",
    "vezes": <número inteiro ou null>,
    "entrega_ou_loja": "<valores possíveis: entrega, motoboy, correios, loja, presencial ou null>",
    "rastreio": "<código de rastreio se houver, ou null>",
    "obs": "<observações adicionais ou null>"
  }
}

REGRAS CRÍTICAS DE RETORNO:
1. Não adicione markdown como ```json ou textos explicativos antes ou depois. Retorne APENAS o objeto JSON puro.
2. Limpe os valores numéricos. Por exemplo, se disser "R$ 23" ou "23 reais", extraia apenas o valor numérico 23.0.
3. Se o método de pagamento for "credito", tente identificar o número de "vezes". Se for "fiado", tente identificar se há "valor_restante".
"""

### Bloco 4: Validação de Dados e Regras de Negócio


In [ ]:
def validar_e_converter_dados(dados):
    # 1. Verificar se retornou as 11 chaves do JSON
    chaves_esperadas = {
        "data", "oq_vendeu", "pra_quem", "valor", "valor_restante",
        "como_pagou", "cartao", "vezes", "entrega_ou_loja", "rastreio", "obs"
    }

    if not isinstance(dados, dict):
        return False, "O retorno da inteligência artificial não é um dicionário estruturado."

    chaves_existentes = set(dados.keys())
    if not chaves_esperadas.issubset(chaves_existentes):
        chaves_faltantes = chaves_esperadas - chaves_existentes
        return False, f"Contrato de dados violado. Chaves ausentes: {list(chaves_faltantes)}"

    # 2. Dados Obrigatórios Básicos
    obrigatorios = ["data", "oq_vendeu", "valor", "como_pagou", "entrega_ou_loja"]
    erros_obrigatorios = []
    for campo in obrigatorios:
        if dados.get(campo) is None or str(dados.get(campo)).strip() == "":
            erros_obrigatorios.append(campo)

    if erros_obrigatorios:
        return False, f"⚠️ Faltam dados obrigatórios para concluir a venda: {', '.join(erros_obrigatorios)}. Por favor, informe-os para que eu possa salvar."

    # 3. Conferir e converter tipos
    # Converter valor para Decimal
    try:
        dados["valor"] = Decimal(str(dados["valor"]))
    except (InvalidOperation, TypeError, ValueError):
        return False, "❌ O campo 'valor' deve ser um número decimal válido."

    # Converter valor_restante para Decimal (se preenchido)
    if dados.get("valor_restante") is not None:
        try:
            dados["valor_restante"] = Decimal(str(dados["valor_restante"]))
        except (InvalidOperation, TypeError, ValueError):
            return False, "❌ O campo 'valor_restante' deve ser um número decimal válido."
    else:
        dados["valor_restante"] = Decimal('0.00')

    # Converter vezes para inteiro (se preenchido)
    if dados.get("vezes") is not None:
        try:
            dados["vezes"] = int(float(str(dados["vezes"])))
        except (TypeError, ValueError):
            return False, "❌ O campo 'vezes' deve ser um número inteiro válido."

    # Normalização de tipos textuais
    campos_texto = ["oq_vendeu", "pra_quem", "como_pagou", "entrega_ou_loja", "rastreio", "obs", "cartao"]
    for campo in campos_texto:
        if dados.get(campo) is not None:
            dados[campo] = str(dados[campo]).strip()

    # 4. Regras condicionais específicas
    como_pagou = str(dados["como_pagou"]).lower()

    # Se como_pagou for 'credito', obrigatorio preencher 'vezes'
    if como_pagou == "credito":
        if dados.get("vezes") is None or dados.get("vezes") <= 0:
            return False, "⚠️ O pagamento foi feito no Crédito, portanto preciso que informe em quantas **vezes** foi parcelado."

    # Se como_pagou for 'fiado', obrigatorio preencher 'valor_restante'
    if como_pagou == "fiado":
        if dados.get("valor_restante") is None or dados.get("valor_restante") <= 0:
            return False, "⚠️ A venda foi marcada como Fiado, por favor informe o **valor restante** a ser pago."

    return True, dados

### Bloco 5: Integração com a Planilha Excel
Função responsável por ler o arquivo `Vendas_da_maria.xlsx`, adicionar o novo registro validado e salvar.

In [ ]:
def salvar_na_planilha(dados_validados):
    colunas_mapeamento = {
        'data': 'data',
        'oq_vendeu': 'oq vendeu',
        'pra_quem': 'pra quem',
        'valor': 'valor',
        'como_pagou': 'como pagou',
        'cartao': 'cartao',
        'vezes': 'vezes',
        'entrega_ou_loja': 'entrega ou loja',
        'rastreio': 'rastreio',
        'obs': 'obs'
    }

    # Carrega ou cria a planilha excel
    if os.path.exists(PLANILHA_NOME):
        df_excel = pd.read_excel(PLANILHA_NOME)
        print(df_excel.head())
    else:
        df_excel = pd.DataFrame(columns=list(colunas_mapeamento.values()) + ['valor_restante'])

    # Prepara a nova linha
    nova_linha = {}
    for chave_json, nome_coluna_excel in colunas_mapeamento.items():
        nova_linha[nome_coluna_excel] = dados_validados.get(chave_json)

    # Se a coluna 'valor_restante' ainda não existir na planilha física, criamos
    nova_linha['valor_restante'] = float(dados_validados.get('valor_restante', 0))

    # Garante conversões amigáveis para o Pandas salvar no Excel
    nova_linha['valor'] = float(dados_validados['valor'])

    df_nova_linha = pd.DataFrame([nova_linha])
    df_final = pd.concat([df_excel, df_nova_linha], ignore_index=True)

    df_final.to_excel(PLANILHA_NOME, index=False)
    print(f"Venda registrada no Excel com sucesso: {nova_linha}")

In [ ]:
def consultar_na_planilha(filtros):
    """
    Busca registros na planilha Excel aplicando múltiplos filtros estruturados de forma combinada.
    Aceita um dicionário de filtros contendo 'cliente', 'produto', 'data', 'como_pagou' ou 'busca_geral'.
    """
    import os
    import pandas as pd

    if not os.path.exists(PLANILHA_NOME):
        return "⚠️ A planilha de vendas ainda não foi criada ou não possui registros salvos."

    try:
        df = pd.read_excel(PLANILHA_NOME)
    except Exception as e:
        return f"❌ Erro ao ler a planilha: {e}"

    if df.empty:
        return "⚠️ A planilha está vazia no momento."

    if not isinstance(filtros, dict):
        return "⚠️ Filtros de busca inválidos."

    # Começamos com uma máscara que aceita tudo, e vamos estreitando com as condições que a IA nos fornecer
    mascara = pd.Series(True, index=df.index)
    filtros_aplicados = []

    # 1. Filtro por Cliente (Pra quem)
    cliente = filtros.get("cliente")
    if cliente:
        col = "pra quem"
        if col in df.columns:
            mascara &= df[col].astype(str).str.lower().str.contains(str(cliente).lower(), na=False)
            filtros_aplicados.append(f"👤 Cliente: *{cliente}*")

    # 2. Filtro por Produto (Oq vendeu)
    produto = filtros.get("produto")
    if produto:
        col = "oq vendeu"
        if col in df.columns:
            mascara &= df[col].astype(str).str.lower().str.contains(str(produto).lower(), na=False)
            filtros_aplicados.append(f"🛍️ Produto: *{produto}*")

    # 3. Filtro por Data
    data_filtro = filtros.get("data")
    if data_filtro:
        col = "data"
        if col in df.columns:
            mascara &= df[col].astype(str).str.contains(str(data_filtro), na=False)
            filtros_aplicados.append(f"📅 Data: *{data_filtro}*")

    # 4. Filtro por Forma de Pagamento
    como_pagou = filtros.get("como_pagou")
    if como_pagou:
        col = "como pagou"
        if col in df.columns:
            mascara &= df[col].astype(str).str.lower() == str(como_pagou).lower()
            filtros_aplicados.append(f"💳 Pagamento: *{como_pagou}*")

    # 5. Filtro Geral (Busca genérica por termos soltos nas colunas principais)
    busca_geral = filtros.get("busca_geral")
    if busca_geral:
        termo_geral = str(busca_geral).lower()
        mascara_geral = pd.Series(False, index=df.index)
        colunas_busca = ['data', 'oq vendeu', 'pra quem', 'como pagou', 'obs']
        for col in colunas_busca:
            if col in df.columns:
                mascara_geral |= df[col].astype(str).str.lower().str.contains(termo_geral, na=False)
        mascara &= mascara_geral
        filtros_aplicados.append(f"🔍 Termo geral: *{busca_geral}*")

    # Se nenhum filtro foi de fato fornecido pela IA
    if not filtros_aplicados:
        return "⚠️ Não identifiquei critérios ou termos de busca na sua solicitação. Tente algo como 'vendas de ontem' ou 'vendas para Maria'."

    # Aplica o filtro consolidado
    resultados = df[mascara]

    if resultados.empty:
        titulos_filtros = ", ".join(filtros_aplicados)
        return f"🔍 Não encontrei nenhuma venda correspondente aos filtros:\n{titulos_filtros}."

    # Formatação da mensagem final para o usuário do Telegram
    resposta = "🔍 *Resultados da Pesquisa Estruturada:*\n"
    resposta += "\n".join([f" - {f}" for f in filtros_aplicados]) + "\n\n"

    for i, (_, row) in enumerate(resultados.head(15).iterrows(), 1):
        data_val = row.get('data', 'N/A')
        prod_val = row.get('oq vendeu', 'N/A')
        cli_val = row.get('pra quem', 'N/A')
        valor_val = row.get('valor', '0.00')
        pag_val = row.get('como pagou', 'N/A')
        obs_val = row.get('obs', '')
        restante_val = row.get('valor_restante', 0.0)

        resposta += f"*{i}.* 📅 `{data_val}` | 🛍️ *{prod_val}* | 👤 *{cli_val}*\n"
        resposta += f"    💵 R$ {valor_val} | 💳 {pag_val}"
        if float(restante_val) > 0:
            resposta += f" (Restante: R$ {restante_val})"
        if pd.notna(obs_val) and str(obs_val).strip():
            resposta += f"\n    📝 _Obs: {obs_val}_"
        resposta += "\n\n"

    if len(resultados) > 15:
        resposta += f"⚠️ _Mostrando apenas as primeiras 15 de {len(resultados)} correspondências._"

    return resposta

### Bloco 6: Processamento de Inteligência Artificial e Resolução de Intenções

In [ ]:
import time

def processar_mensagem_com_ia(texto_entrada):
    # Adicionamos uma política de tentativas (retries) com tempo de espera para lidar com erros 503 (serviço indisponível)
    tentativas_maximas = 3
    tempo_espera = 2

    for tentativa in range(1, tentativas_maximas + 1):
        try:
            # Gemini 3.6 Flash para organizar as informações no formato JSON estruturado
            response = gemini_client.models.generate_content(
                model='gemini-3.6-flash',
                contents=PROMPT_CONTRATO_DADOS + f"\n\nTexto do Usuário: '{texto_entrada}'"
            )

            texto_resposta = response.text.strip()
            # Remove eventuais marcações de markdown de bloco de código
            texto_resposta = re.sub(r'^```json\s*|\s*```$', '', texto_resposta, flags=re.MULTILINE)

            resultado_json = json.loads(texto_resposta)
            return resultado_json

        except Exception as e:
            print(f"⚠️ Tentativa {tentativa}/{tentativas_maximas} falhou ao estruturar dados com a IA: {e}")
            if tentativa < tentativas_maximas:
                time.sleep(tempo_espera * tentativa) # Recuo exponencial simples
            else:
                print("❌ Limite de tentativas alcançado. Não foi possível conectar ao Gemini no momento.")
                return None

### Bloco 7: Handlers do Telegram (Recebimento de Áudio e Texto)


In [ ]:
# MENSAGEM DE TEXTO
@bot.message_handler(content_types=['text'])
def responder_texto(mensagem):
    processar_e_responder(mensagem, mensagem.text)

# MENSAGEM DE ÁUDIO
@bot.message_handler(content_types=['voice'])
def responder_audio(mensagem):
    ogg_filename = f"audio_{mensagem.chat.id}.ogg"
    wav_filename = f"audio_{mensagem.chat.id}.wav"

    try:
        bot.reply_to(mensagem, "🎙️ Áudio recebido! Transcrevendo com Gemini...")

        # 1. Baixa o arquivo de áudio recebido
        file_info = bot.get_file(mensagem.voice.file_id)
        downloaded_file = bot.download_file(file_info.file_path)

        with open(ogg_filename, 'wb') as new_file:
            new_file.write(downloaded_file)

        # 2. Converte OGG para WAV localmente
        data, samplerate = sf.read(ogg_filename)
        sf.write(wav_filename, data, samplerate, format='WAV', subtype='PCM_16')

        # 3. Envia o arquivo de áudio WAV para a API Files do Gemini
        audio_api_file = gemini_client.files.upload(
            file=wav_filename,
            config=types.UploadFileConfig(mime_type="audio/wav")
        )

        # 4. Transcreve com gemini-3.5-transcribe passando a configuração como dicionário simples
        transcricao = gemini_client.interactions.create(
            model="gemini-3.5-transcribe",
            input=[{"type": "audio", "uri": audio_api_file.uri, "mime_type": audio_api_file.mime_type}],
            generation_config={
                "transcription_config": {
                    "mode": "smart",
                    "language_codes": ["pt-BR"]
                }
            }
        )

        texto_transcrito = transcricao.output_text
        bot.send_message(mensagem.chat.id, f"📝 *Transcrição:*\n\"{texto_transcrito}\"", parse_mode="Markdown")

        # Limpeza do arquivo na nuvem da API do Gemini
        gemini_client.files.delete(name=audio_api_file.name)

        # Envia o texto obtido do áudio para o fluxo de processamento
        processar_e_responder(mensagem, texto_transcrito)

    except Exception as e:
        bot.send_message(mensagem.chat.id, f"❌ Erro no processamento de voz: {e}")

    finally:
        # Limpa os arquivos temporários criados no Colab
        if os.path.exists(ogg_filename):
            os.remove(ogg_filename)
        if os.path.exists(wav_filename):
            os.remove(wav_filename)


def processar_e_responder(mensagem, texto):
    # Solicita à IA que estruture a mensagem
    estrutura = processar_mensagem_com_ia(texto)

    if not estrutura:
        bot.reply_to(mensagem, "❌ Não consegui processar a estrutura desta mensagem. Tente reescrevê-la de forma simples.")
        return

    intencao = estrutura.get("intencao")

    # Roteamento de Intenção
    if intencao == "consultar":
        termo_busca = estrutura.get("busca", "")
        bot.reply_to(mensagem, "🔍 Buscando informações na planilha, um momento...")

        # Realiza a busca no Excel chamando a função correspondente
        resultado_busca = consultar_na_planilha(termo_busca)
        bot.send_message(mensagem.chat.id, resultado_busca, parse_mode="Markdown")

    elif intencao == "inserir":
        dados_brutos = estrutura.get("dados", {})
        sucesso, resultado = validar_e_converter_dados(dados_brutos)

        if sucesso:
            # Se passou na validação, salva na planilha
            salvar_na_planilha(resultado)
            resumo_venda = f"✅ **Dados inseridos com sucesso!**\n\n"
            resumo_venda += f"📅 Data: {resultado.get('data')}\n"
            resumo_venda += f"🛍️ Produto: {resultado.get('oq_vendeu')}\n"
            resumo_venda += f"👤 Cliente: {resultado.get('pra_quem')}\n"
            resumo_venda += f"💵 Valor: R$ {resultado.get('valor')}\n"
            resumo_venda += f"💳 Pagamento: {resultado.get('como_pagou')}"
            if resultado.get('vezes'):
                resumo_venda += f" ({resultado.get('vezes')}x)"

            bot.reply_to(mensagem, resumo_venda, parse_mode="Markdown")
        else:
            # Se faltar dado ou falhar na validação, envia a mensagem solicitando os ajustes
            bot.reply_to(mensagem, resultado, parse_mode="Markdown")
    else:
        bot.reply_to(mensagem, "Não entendi muito bem se você deseja consultar ou inserir. Pode especificar melhor?")

print("Eventos de escuta do Bot configurados com pesquisa integrada real!")

In [ ]:
def processar_e_responder(mensagem, texto):
    # Solicita à IA que estruture a mensagem
    estrutura = processar_mensagem_com_ia(texto)

    if not estrutura:
        bot.reply_to(mensagem, "❌ Não consegui processar a estrutura desta mensagem. Tente reescrevê-la de forma simples.")
        return

    intencao = estrutura.get("intencao")

    # Roteamento de Intenção
    if intencao == "consultar":
        filtros_busca = estrutura.get("filtros", {})
        bot.reply_to(mensagem, "🔍 Analisando os parâmetros de busca e lendo a planilha, um momento...")

        # Realiza a busca utilizando os múltiplos filtros estruturados no Excel
        resultado_busca = consultar_na_planilha(filtros_busca)
        bot.send_message(mensagem.chat.id, resultado_busca, parse_mode="Markdown")

    elif intencao == "inserir":
        dados_brutos = estrutura.get("dados", {})
        sucesso, resultado = validar_e_converter_dados(dados_brutos)

        if sucesso:
            # Se passou na validação, salva na planilha
            salvar_na_planilha(resultado)
            resumo_venda = f"✅ **Dados inseridos com sucesso!**\n\n"
            resumo_venda += f"📅 Data: {resultado.get('data')}\n"
            resumo_venda += f"🛍️ Produto: {resultado.get('oq_vendeu')}\n"
            resumo_venda += f"👤 Cliente: {resultado.get('pra_quem')}\n"
            resumo_venda += f"💵 Valor: R$ {resultado.get('valor')}\n"
            resumo_venda += f"💳 Pagamento: {resultado.get('como_pagou')}"
            if resultado.get('vezes'):
                resumo_venda += f" ({resultado.get('vezes')}x)"

            bot.reply_to(mensagem, resumo_venda, parse_mode="Markdown")
        else:
            # Se faltar dado ou falhar na validação, envia a mensagem solicitando os ajustes
            bot.reply_to(mensagem, resultado, parse_mode="Markdown")
    else:
        bot.reply_to(mensagem, "Não entendi muito bem se você deseja consultar ou inserir. Pode especificar melhor?")

print("Função operacional processar_e_responder configurada com filtros de busca robustos estruturados!")

### Bloco 8: Ativação do Bot

In [ ]:
import requests
import time

# Força a desconexão de instâncias concorrentes fechando o polling anterior via API oficial
try:
    print("Enviando comando para fechar instâncias e deletar Webhook...")
    close_url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/close"
    delete_webhook_url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/deleteWebhook?drop_pending_updates=True"

    res_close = requests.post(close_url).json()
    res_del = requests.post(delete_webhook_url).json()

    print(f"Resposta close: {res_close}")
    print(f"Resposta deleteWebhook: {res_del}")
    time.sleep(3)
except Exception as e:
    print(f"Erro ao limpar conexões via HTTP request: {e}")

# Tenta iniciar o polling novamente de forma limpa
try:
    print("Iniciando escuta do bot de forma limpa...")
    bot.remove_webhook()
    bot.infinity_polling(timeout=10, long_polling_timeout=5)
except Exception as e:
    print(f"Erro no polling: {e}")